In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.path as mpath
from matplotlib.patches import Rectangle
from scipy import stats
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import generic_filter, binary_dilation
# shapely contains_xy is the modern API (shapely ≥ 2.0); fall back to
# the deprecated vectorized.contains for older shapely.
try:
    from shapely import contains_xy as _shp_contains_xy
    def _shp_contains(geom, x, y): return _shp_contains_xy(geom, x, y)
except ImportError:
    from shapely.vectorized import contains as _shp_contains
from netCDF4 import Dataset, num2date
import geopandas as gpd

warnings.filterwarnings('ignore')

# ── Directories ────────────────────────────────────────────────────────
DATA_DIR  = "data/"
MODEL_DIR = "models/"
os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Region: Northern Indian Ocean 75-100E, 5S-35N ─────────────────────
IOD_LAT_MIN, IOD_LAT_MAX = -5,  35
IOD_LON_MIN, IOD_LON_MAX = 75, 100
WEST_POLE = {"lat": (10, 25), "lon": (75,  85)}   # Arabian Sea
EAST_POLE = {"lat": (10, 25), "lon": (85, 100)}   # Bay of Bengal

# ── GPU setup ──────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, ConvLSTM2D, Conv2D,
                                      BatchNormalization, Dropout)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    tf.config.optimizer.set_jit(True)
    print(f"✅ GPU ready: {[g.name for g in gpus]} | mixed_float16 + XLA")
else:
    print("⚠️  No GPU — running on CPU")

# ── Figure dimensions ──────────────────────────────────────────────────
MAP_FIGW = 7.0

# ══════════════════════════════════════════════════════════════════════
# SHAPEFILE SETUP
# ══════════════════════════════════════════════════════════════════════
SHAPEFILE_PATH = "data/india_st.shp"

from shapely.geometry import box as shapely_box

_raw_gdf = gpd.read_file(SHAPEFILE_PATH)
if _raw_gdf.crs is None:
    print("⚠️  Shapefile has no CRS — assuming EPSG:4326 (WGS84)")
    _raw_gdf = _raw_gdf.set_crs("EPSG:4326")
else:
    _raw_gdf = _raw_gdf.to_crs("EPSG:4326")

india_states_gdf = _raw_gdf
india_outer_gdf  = india_states_gdf.dissolve()

print(f"✅ India shapefile loaded — {len(india_states_gdf)} states | "
      f"bounds: {india_states_gdf.total_bounds.round(2)}")

# ── Global land shapefile (Natural Earth 50m) ─────────────────────────
# Used as the authoritative land mask so EVERY coastline is masked,
# not just India's. Auto-downloaded once and cached in DATA_DIR.
GLOBAL_LAND_PATH = os.path.join(DATA_DIR, "ne_50m_land.shp")
# Multi-mirror list — tried in order until one works.
GLOBAL_LAND_URLS = [
    "https://naciscdn.org/naturalearth/50m/physical/ne_50m_land.zip",
    "https://naturalearth.s3.amazonaws.com/50m_physical/ne_50m_land.zip",
    "https://www.naturalearthdata.com/http//www.naturalearthdata.com/download/50m/physical/ne_50m_land.zip",
]

def _ensure_global_land():
    if os.path.exists(GLOBAL_LAND_PATH):
        return GLOBAL_LAND_PATH
    import urllib.request, zipfile, ssl
    zip_path = os.path.join(DATA_DIR, "ne_50m_land.zip")
    ctx = ssl.create_default_context()
    headers = {"User-Agent": "Mozilla/5.0 (compatible; SST-notebook/1.0)"}
    for url in GLOBAL_LAND_URLS:
        try:
            print(f"Downloading Natural Earth 50m land from {url.split('/')[2]} …",
                  end=" ", flush=True)
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=60, context=ctx) as r, \
                 open(zip_path, "wb") as f:
                f.write(r.read())
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(DATA_DIR)
            os.remove(zip_path)
            print("done ✅")
            return GLOBAL_LAND_PATH
        except Exception as e:
            print(f"failed ({type(e).__name__})")
            continue
    print("⚠️  All mirrors failed.")
    print("    Manual download: https://www.naturalearthdata.com/downloads/50m-physical-vectors/")
    print("    Unzip ne_50m_land.zip into the data/ directory and re-run this cell.")
    return None

_global_path = _ensure_global_land()
if _global_path is not None:
    global_land_gdf = gpd.read_file(_global_path)
    if global_land_gdf.crs is None:
        global_land_gdf = global_land_gdf.set_crs("EPSG:4326")
    else:
        global_land_gdf = global_land_gdf.to_crs("EPSG:4326")
    print(f"✅ Global land shapefile loaded — {len(global_land_gdf)} polygons")
else:
    print("⚠️  Global mask unavailable — falling back to India-only mask")
    global_land_gdf = india_outer_gdf

# ── Zoom extent from shapefile bounds + ocean buffer ──────────────────
_b            = india_states_gdf.total_bounds
_OCEAN_BUFFER = 3.0
_SHP_LON_MIN  = _b[0] - _OCEAN_BUFFER
_SHP_LON_MAX  = _b[2] + _OCEAN_BUFFER
_SHP_LAT_MIN  = _b[1] - _OCEAN_BUFFER
_SHP_LAT_MAX  = _b[3] + _OCEAN_BUFFER

_lon_span = _SHP_LON_MAX - _SHP_LON_MIN
_lat_span = _SHP_LAT_MAX - _SHP_LAT_MIN
MAP_FIGH  = round(MAP_FIGW * _lat_span / _lon_span, 1)
print(f"   Zoom extent : lon [{_SHP_LON_MIN:.1f}, {_SHP_LON_MAX:.1f}]  "
      f"lat [{_SHP_LAT_MIN:.1f}, {_SHP_LAT_MAX:.1f}]")
print(f"   Figure size : {MAP_FIGW} × {MAP_FIGH} inches")

# ── NaN pre-fill for coarse grids ─────────────────────────────────────
def _fill_nan_coarse(data, passes=12):
    """
    Fill NaN pixels in a 2-D coarse array with the mean of valid neighbours.
    Runs `passes` iterations so isolated NaN pockets are fully filled.
    This is applied BEFORE interpolation so the fine grid never sees NaN
    at valid ocean locations — only the shapefile mask sets land to NaN.
    """
    filled = data.copy().astype(np.float64)
    def _nbr_mean(vals):
        v = vals[~np.isnan(vals)]
        return np.mean(v) if len(v) else np.nan
    for _ in range(passes):
        if not np.isnan(filled).any():
            break
        tmp = generic_filter(filled, _nbr_mean, size=3, mode='nearest')
        nan_px = np.isnan(filled)
        filled[nan_px] = tmp[nan_px]
    return filled.astype(np.float32)

# ── Land-mask rasterisation (robust for small islands) ───────────────
def _rasterise_land(gdf, lon_f, lat_f):
    """
    Build a boolean land-mask on the fine grid using shapely.contains_xy.
    Handles MultiPolygons, holes, and tiny island polygons reliably.
    Spatial-index clip (.cx) keeps the global Natural Earth gdf fast —
    only polygons inside the fine-grid bbox are evaluated.
    """
    Lg, Pg = np.meshgrid(lon_f, lat_f)
    minx, maxx = float(lon_f.min()), float(lon_f.max())
    miny, maxy = float(lat_f.min()), float(lat_f.max())
    try:
        gdf_local = gdf.cx[minx:maxx, miny:maxy]
    except Exception:
        gdf_local = gdf
    land = np.zeros(Lg.shape, dtype=bool)
    for geom in gdf_local.geometry:
        if geom is None or geom.is_empty:
            continue
        land |= _shp_contains(geom, Lg, Pg)
    return land

_land_cache = {}

def _get_land_mask(lon, lat, factor=20):
    key = (round(float(lon.min()),3), round(float(lon.max()),3),
           round(float(lat.min()),3), round(float(lat.max()),3),
           len(lon), len(lat), factor)
    if key not in _land_cache:
        lon_f = np.linspace(float(lon.min()), float(lon.max()), len(lon)*factor)
        lat_f = np.linspace(float(lat.min()), float(lat.max()), len(lat)*factor)
        print("  ↳ Rasterising global land onto fine grid (one-time)…", end=" ", flush=True)
        mask = _rasterise_land(global_land_gdf, lon_f, lat_f)
        _land_cache[key] = (mask, lon_f, lat_f)
        print("done ✅")
    return _land_cache[key]

# ── smooth_plot ────────────────────────────────────────────────────────
def smooth_plot(ax, lon, lat, data, cmap, vmin=None, vmax=None, factor=20,
                add_coast=True, ocean_mask=None):
    """
    SST / trend map. Land everywhere on Earth is masked from the
    Natural Earth 50m global land shapefile, so coastlines (India,
    Sri Lanka, Bangladesh, Myanmar, Pakistan, Tibet, Andaman/Nicobar,
    Lakshadweep, ...) are all clipped accurately.

    Pipeline:
      1. Pre-fill NaN in coarse data with neighbour means so interpolation
         is smooth across the SST file's 1° land halo.
      2. Linear-interpolate onto a fine grid.
      3. Mask the fine grid with the global land shapefile → gradient
         exists only over true ocean.
      4. State-boundary overlay (from India shapefile) + zoom to India.

    `ocean_mask` is accepted for backward compatibility but is no longer
    needed — the global land mask handles all coastlines cleanly.
    """
    land_fine, lon_f, lat_f = _get_land_mask(lon, lat, factor)
    nlat_f, nlon_f = len(lat_f), len(lon_f)
    Lg, Pg = np.meshgrid(lon_f, lat_f)

    # Step 1: pre-fill NaN so interpolation is clean across the SST 1° halo
    data_filled = _fill_nan_coarse(data)

    # Step 2: interpolate value field onto fine grid
    sst_up = RegularGridInterpolator(
        (lat, lon), data_filled.astype(float),
        method="linear", bounds_error=False, fill_value=np.nan
    )((Pg, Lg))

    # Step 3: GLOBAL land mask — authoritative; handles every coastline
    sst_up[land_fine] = np.nan

    # Step 4: render gradient over ocean
    kw = dict(cmap=cmap, shading="gouraud", zorder=1)
    if vmin is not None: kw["vmin"] = vmin
    if vmax is not None: kw["vmax"] = vmax
    im = ax.pcolormesh(lon_f, lat_f, sst_up, **kw)

    # Step 5: grey land fill from global mask
    rgba = np.zeros((nlat_f, nlon_f, 4), dtype=np.float32)
    rgba[land_fine] = [0.72, 0.72, 0.72, 1.0]
    ax.imshow(rgba,
              extent=[float(lon.min()), float(lon.max()),
                      float(lat.min()), float(lat.max())],
              origin="lower", aspect="auto",
              interpolation="bicubic", zorder=2)

    # Step 6: boundary lines (state lines + India outer)
    if add_coast:
        india_states_gdf.boundary.plot(            # thin state lines
            ax=ax, color="#555555", linewidth=0.4,
            linestyle="--", zorder=3)
        india_outer_gdf.boundary.plot(             # bold India outline
            ax=ax, color="#111111", linewidth=1.4, zorder=4)

    # Step 7: zoom to India + ocean buffer (adjustable=box silences the
    # "Ignoring fixed y limits…" warning)
    x_min = max(float(lon.min()), _SHP_LON_MIN)
    x_max = min(float(lon.max()), _SHP_LON_MAX)
    y_min = max(float(lat.min()), _SHP_LAT_MIN)
    y_max = min(float(lat.max()), _SHP_LAT_MAX)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    return im

# ── Extract IOD grid ───────────────────────────────────────────────────
def extract_iod_grid(path):
    nc    = Dataset(path, "r")
    lat   = np.array(nc.variables["lat"][:], dtype=np.float32)
    lon   = np.array(nc.variables["lon"][:], dtype=np.float32)
    sst   = np.ma.filled(nc.variables["sst"][:].astype(np.float32), np.nan)
    t     = nc.variables["time"]
    dates = pd.to_datetime([str(d) for d in num2date(t[:], units=t.units,
                                                      calendar="standard")])
    nc.close()
    if lon.min() < 0:
        lon = np.where(lon < 0, lon + 360, lon)
    lat_mask = (lat >= IOD_LAT_MIN) & (lat <= IOD_LAT_MAX)
    lon_mask = (lon >= IOD_LON_MIN) & (lon <= IOD_LON_MAX)
    return (sst[:, lat_mask, :][:, :, lon_mask].astype(np.float32),
            lat[lat_mask], lon[lon_mask], dates)

grid, lat, lon, dates = extract_iod_grid(f"{DATA_DIR}sst.mnmean.nc")
print(f"✅ {grid.shape[0]} months | {grid.shape[1]}x{grid.shape[2]} grid | "
      f"{dates[0].strftime('%Y-%m')} → {dates[-1].strftime('%Y-%m')}")


In [ ]:
# ── CHANGE 1 ── Cell 3: 2D climatology from 3D dataset
# grid shape is (time, lat, lon) — 3D.
# np.nanmean(axis=(1,2)) collapses lat+lon into a single basin-wide mean per month.
# Result: 1D time series → grouped by calendar month → 12-value climatology.

df_sst = pd.DataFrame({'Date': dates, 'SST': np.nanmean(grid, axis=(1, 2))})
df_sst['Month'] = df_sst['Date'].dt.month
df_sst['Year']  = df_sst['Date'].dt.year

climatology = df_sst.groupby('Month')['SST'].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(month_names, climatology.values, marker='o', color='#d95f02', linewidth=2)
ax.set(title=f"Seasonal Climatology: Average Indian Ocean SST by Month\n(averaged across all {len(dates)} months, {dates[0].year}–{dates[-1].year})",
       xlabel="Month", ylabel="Average SST (°C)")
ax.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
plt.tight_layout()
plt.show()
print(f"Note: 3D grid ({grid.shape}) → spatial mean → 1D series → grouped by month → 12-point climatology")

In [ ]:
# ── CHANGE 2 ── Cell 4: DMI formula and interpretation
# DMI = mean SST (Arabian Sea: 75-85E, 10N-25N) - mean SST (Bay of Bengal: 85-100E, 10N-25N)
# Positive DMI → warm west, cool east → Positive IOD event
# Negative DMI → cool west, warm east → Negative IOD event

def pole_mask(lat, lon, pole):
    return np.ix_((lat >= pole["lat"][0]) & (lat <= pole["lat"][1]),
                  (lon >= pole["lon"][0]) & (lon <= pole["lon"][1]))

wi, ei = pole_mask(lat, lon, WEST_POLE), pole_mask(lat, lon, EAST_POLE)
dmi_series = (np.nanmean(grid[:, wi[0], wi[1]], axis=(1, 2)) -
              np.nanmean(grid[:, ei[0], ei[1]], axis=(1, 2)))
dmi_smooth = pd.Series(dmi_series).rolling(window=3, center=True).mean()
years_frac  = df_sst['Year'] + df_sst['Month'] / 12

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(years_frac, dmi_smooth, color='black', linewidth=1, label="3-Month Smooth DMI")
ax.fill_between(years_frac, dmi_smooth, 0, where=(dmi_smooth >= 0),
                color='#d73027', alpha=0.6, label="Positive IOD (warm west, cool east)")
ax.fill_between(years_frac, dmi_smooth, 0, where=(dmi_smooth  < 0),
                color='#4575b4', alpha=0.6, label="Negative IOD (cool west, warm east)")
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.axhline( 0.5, color='#d73027', linewidth=0.8, linestyle=':', alpha=0.6)
ax.axhline(-0.5, color='#4575b4', linewidth=0.8, linestyle=':', alpha=0.6)
ax.text(years_frac.iloc[-1], 0.52, '+0.5 threshold', fontsize=8, color='#d73027', ha='right')
ax.text(years_frac.iloc[-1], -0.62, '−0.5 threshold', fontsize=8, color='#4575b4', ha='right')
ax.set(title="Historical Dipole Mode Index  |  DMI = SST(West Pole) − SST(East Pole)",
       xlabel="Year", ylabel="DMI (°C)")
ax.legend()
plt.tight_layout()
plt.show()

pos_events = (dmi_smooth > 0.5).sum()
neg_events = (dmi_smooth < -0.5).sum()
print(f"Months with positive IOD (DMI > +0.5°C): {pos_events}")
print(f"Months with negative IOD (DMI < −0.5°C): {neg_events}")

In [ ]:
# ── Interannual SST Variability ────────────────────────────────────────
# True interannual variability = std of MONTHLY ANOMALIES (seasonal cycle removed).
# Using np.nanstd ignores missing time steps per pixel so the ocean is not
# wiped out by NaN propagation.

# Step 1: build monthly climatology (12 maps, one per calendar month)
month_idx = np.array([d.month - 1 for d in dates])
clim_maps = np.array([np.nanmean(grid[month_idx == m], axis=0) for m in range(12)])

# Step 2: monthly anomalies (deseasoned)
anom = grid - clim_maps[month_idx]

# Step 3: interannual std at each pixel (NaN-safe)
std_map = np.nanstd(anom, axis=0)

ocean_mask_full = ~np.isnan(np.nanmean(grid, axis=0))

fig, ax = plt.subplots(figsize=(MAP_FIGW, MAP_FIGH))
im = smooth_plot(ax, lon, lat, std_map, cmap="plasma", ocean_mask=ocean_mask_full)
plt.colorbar(im, ax=ax).set_label("Std Dev (°C)")
for pole, label in [(WEST_POLE, "West Pole"), (EAST_POLE, "East Pole")]:
    ax.add_patch(Rectangle((pole["lon"][0], pole["lat"][0]),
                            pole["lon"][1] - pole["lon"][0], pole["lat"][1] - pole["lat"][0],
                            linewidth=1.5, edgecolor='white', facecolor='none', linestyle='--'))
    ax.text(pole["lon"][0] + 2, pole["lat"][0] + 2, label, color='white', fontsize=10, weight='bold')
ax.set(title="Interannual SST Variability (deseasoned)", xlabel="Longitude (°E)", ylabel="Latitude")
plt.tight_layout()
plt.show()
print(f"Interannual std range over ocean: "
      f"{np.nanmin(std_map[ocean_mask_full]):.3f} – {np.nanmax(std_map[ocean_mask_full]):.3f} °C")


In [ ]:
# ── CHANGE 3 ── Cell 6: User-configurable decade window
# Pixel-wise OLS slope = Σ(xm_i × y_i) / Σ(xm_i²) using ONLY valid time steps
# per pixel (NaN-robust). Multiplied by 120 = 10 yr × 12 mo → °C / decade.
#
# ════════════════════════════════════════════════
#  ENTER YOUR DESIRED PERIOD HERE
START_YEAR  = 1981   # ← change to your desired start year
START_MONTH = 1      # ← change to your desired start month (1–12)
END_YEAR    = 2025   # ← change to your desired end year
END_MONTH   = 12     # ← change to your desired end month (1–12)
# ════════════════════════════════════════════════

start_dt = pd.Timestamp(year=START_YEAR, month=START_MONTH, day=1)
end_dt   = pd.Timestamp(year=END_YEAR,   month=END_MONTH,   day=1)

period_mask  = (dates >= start_dt) & (dates <= end_dt)
grid_period  = grid[period_mask]
n_months     = grid_period.shape[0]

if n_months < 24:
    print(f"⚠️  Only {n_months} months in selected range — need at least 24 for a meaningful trend.")
else:
    # NaN-robust trend: drop NaN time steps per pixel
    xm_p   = (np.arange(n_months, dtype=np.float32) - (n_months - 1) / 2)
    xm_3d  = xm_p[:, None, None]                    # (T, 1, 1)
    valid  = ~np.isnan(grid_period)                  # (T, H, W) bool

    # Anomalies w.r.t. nanmean → still NaN where invalid, but OK
    y_anom = grid_period - np.nanmean(grid_period, axis=0, keepdims=True)

    # Σ(x · y_anom) over valid steps only
    num    = np.where(valid, xm_3d * y_anom, 0.0).sum(axis=0)
    # Σ(x²) over valid steps only — denominator changes per pixel
    denom  = np.where(valid, xm_3d ** 2,   0.0).sum(axis=0)
    n_obs  = valid.sum(axis=0)

    # Need ≥24 valid points; otherwise mark NaN
    trend_decade = np.where((denom > 0) & (n_obs >= 24),
                            num / np.where(denom > 0, denom, 1) * 120,
                            np.nan).astype(np.float32)

    vmax = np.abs(np.nanpercentile(trend_decade, [2, 98])).max()
    fig, ax = plt.subplots(figsize=(MAP_FIGW, MAP_FIGH))
    im = smooth_plot(ax, lon, lat, trend_decade, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                 ocean_mask=(~np.isnan(np.nanmean(grid,axis=0))))
    plt.colorbar(im, ax=ax).set_label("Trend (°C / decade)")
    ax.set(title=f"SST Warming Trend: {START_YEAR}-{START_MONTH:02d} to {END_YEAR}-{END_MONTH:02d}  ({n_months} months)",
           xlabel="Longitude (°E)", ylabel="Latitude")
    plt.tight_layout()
    plt.show()
    print(f"Period: {start_dt.strftime('%b %Y')} → {end_dt.strftime('%b %Y')} | {n_months} months")
    print(f"Max warming: +{np.nanmax(trend_decade):.3f} °C/decade | Max cooling: {np.nanmin(trend_decade):.3f} °C/decade")


In [ ]:
# ── CHANGE 4 ── Cell 7: Mean SST with explicit averaging period
# Uses the full dataset by default. Change MEAN_START_YEAR / MEAN_END_YEAR to restrict.
#
# ════════════════════════════════════════════════
MEAN_START_YEAR = None   # ← set to a year (e.g. 1981) or leave None for full record
MEAN_END_YEAR   = None   # ← set to a year (e.g. 2025) or leave None for full record
# ════════════════════════════════════════════════

if MEAN_START_YEAR is not None or MEAN_END_YEAR is not None:
    s = pd.Timestamp(year=MEAN_START_YEAR or int(dates[0].year),  month=1, day=1)
    e = pd.Timestamp(year=MEAN_END_YEAR   or int(dates[-1].year), month=12, day=1)
    m_mask   = (dates >= s) & (dates <= e)
    grid_avg = grid[m_mask]
    period_label = f"{s.strftime('%b %Y')} – {e.strftime('%b %Y')}  ({m_mask.sum()} months)"
else:
    grid_avg     = grid
    period_label = f"{dates[0].strftime('%b %Y')} – {dates[-1].strftime('%b %Y')}  ({len(dates)} months)"

mean_sst = np.nanmean(grid_avg, axis=0)

fig, ax = plt.subplots(figsize=(MAP_FIGW, MAP_FIGH))
im = smooth_plot(ax, lon, lat, mean_sst, cmap="RdYlBu_r",
                 ocean_mask=(~np.isnan(np.nanmean(grid,axis=0))))
plt.colorbar(im, ax=ax).set_label("Mean SST (°C)")
ax.set(title=f"Mean SST — IOD Spatial Domain\nAveraged over: {period_label}",
       xlabel="Longitude (°E)", ylabel="Latitude")
plt.tight_layout()
plt.show()
print(f"Averaging period: {period_label}")
print(f"Basin mean SST: {np.nanmean(mean_sst):.2f} °C")

In [ ]:
start_idx    = np.where(df_sst['Year'] >= 1981)[0][0]
grid_modern  = grid[start_idx:]
dates_modern = dates[start_idx:]

# Full resolution — no downsampling (small region, 1°×1° grid kept intact)
grid_small = grid_modern.copy()
lat_small  = lat.copy()
lon_small  = lon.copy()
T, H, W    = grid_small.shape

flat   = grid_small.reshape(T, H * W)
filled = (pd.DataFrame(flat)
            .interpolate(method='linear', axis=0)
            .ffill().bfill()
            .values.reshape(T, H, W)
            .astype(np.float32))

sc_min = np.nanmin(filled, axis=0)
sc_max = np.nanmax(filled, axis=0)
rng    = np.where(sc_max != sc_min, sc_max - sc_min, 1.0)
ocean_mask = rng > 0
scaled = np.nan_to_num((filled - sc_min) / rng, nan=0.0)

seq_len        = 12
forecast_steps = 1
n = T - seq_len - forecast_steps + 1

X = np.zeros((n, seq_len, H, W, 1), dtype=np.float32)
y = np.zeros((n,          H, W, 1), dtype=np.float32)
for i in range(n):
    X[i] = scaled[i : i + seq_len, :, :, np.newaxis]
    y[i] = scaled[i + seq_len : i + seq_len + forecast_steps].squeeze(0)[..., np.newaxis]

split_idx = int(n * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
test_dates = dates_modern[seq_len + forecast_steps - 1 + split_idx:]

print(f"✅ Train: {X_train.shape[0]} | Test: {X_test.shape[0]} | Grid: {H}x{W}")

In [ ]:
def build_convlstm(seq_len, H, W, forecast_steps=1):
    inp = Input(shape=(seq_len, H, W, 1))
    x   = ConvLSTM2D(32, (3, 3), padding='same', return_sequences=True)(inp)
    x   = BatchNormalization()(x)
    x   = Dropout(0.2)(x)
    x   = ConvLSTM2D(16, (3, 3), padding='same', return_sequences=False)(x)
    x   = BatchNormalization()(x)
    x   = Dropout(0.2)(x)
    out = Conv2D(forecast_steps, (1, 1), padding='same', activation='linear', dtype='float32')(x)
    m   = Model(inp, out, name='IOD_ConvLSTM')
    m.compile(optimizer=Adam(1e-3), loss='mse', metrics=['mae'])
    return m

model = build_convlstm(seq_len, H, W, forecast_steps)
model.summary()

In [ ]:
n_val = int(len(X_train) * 0.1)
X_tr, X_val = X_train[:-n_val], X_train[-n_val:]
y_tr, y_val = y_train[:-n_val], y_train[-n_val:]

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

history = model.fit(
    X_tr, y_tr, epochs=50, batch_size=16,
    validation_data=(X_val, y_val), callbacks=callbacks, verbose=1
)

fig, ax = plt.subplots(figsize=(8, 4))
ep = range(1, len(history.history['loss']) + 1)
ax.plot(ep, history.history['loss'],     label='Train Loss',      color='#2ca02c', linewidth=2)
ax.plot(ep, history.history['val_loss'], label='Validation Loss', color='#d62728', linestyle='--', linewidth=2)
ax.set(title='Training Curves', xlabel='Epoch', ylabel='MSE')
ax.legend()
ax.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
plt.tight_layout()
plt.show()

model.save(f"{MODEL_DIR}convlstm_iod.keras")
print("✅ Model saved.")

In [ ]:
# ── CHANGE 5 ── Cell 11: Spatial RMSE with inference guide printed
y_pred_scaled = np.clip(model.predict(X_test, batch_size=16)[..., 0], 0.0, 1.0)
y_test_scaled = y_test[..., 0]

y_pred = y_pred_scaled * rng[np.newaxis] + sc_min[np.newaxis]
y_true = y_test_scaled * rng[np.newaxis] + sc_min[np.newaxis]

y_pred[:, ~ocean_mask] = np.nan
y_true[:, ~ocean_mask] = np.nan

rmse = np.sqrt(np.nanmean((y_pred - y_true) ** 2, axis=0))

wi_s = pole_mask(lat_small, lon_small, WEST_POLE)
ei_s = pole_mask(lat_small, lon_small, EAST_POLE)

global_rmse = np.nanmean(rmse)
west_rmse   = np.nanmean(rmse[wi_s])
east_rmse   = np.nanmean(rmse[ei_s])

print(f"➜ Global RMSE : {global_rmse:.3f} °C")
print(f"➜ West Pole   : {west_rmse:.3f} °C  ({'better' if west_rmse < global_rmse else 'worse'} than global average)")
print(f"➜ East Pole   : {east_rmse:.3f} °C  ({'better' if east_rmse < global_rmse else 'worse'} than global average)")
print()
print("Interpretation:")
print(f"  {'✅' if global_rmse < 0.6 else '⚠️ '} Global RMSE {'is competitive (<0.6°C)' if global_rmse < 0.6 else 'is high (>0.6°C) — check inverse scaling'}")
print(f"  {'✅' if east_rmse > west_rmse else '⚠️ '} East Pole RMSE {'correctly higher than West (upwelling variance)' if east_rmse > west_rmse else 'unexpectedly lower than West'}")

fig, ax = plt.subplots(figsize=(MAP_FIGW, MAP_FIGH))
im = smooth_plot(ax, lon_small, lat_small, rmse, cmap='YlOrRd', ocean_mask=ocean_mask)
plt.colorbar(im, ax=ax).set_label('RMSE (°C)')
for pole, name in [(WEST_POLE, "W"), (EAST_POLE, "E")]:
    ax.add_patch(Rectangle((pole["lon"][0], pole["lat"][0]),
                            pole["lon"][1] - pole["lon"][0], pole["lat"][1] - pole["lat"][0],
                            linewidth=1.5, edgecolor='black', facecolor='none', linestyle='--'))
ax.set(title=f'Spatial RMSE Map  |  Global={global_rmse:.3f}°C  West={west_rmse:.3f}°C  East={east_rmse:.3f}°C',
       xlabel='Longitude (°E)', ylabel='Latitude')
plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════
#  ENTER TARGET MONTH TO VISUALISE FORECAST
FORECAST_VIS_YEAR  = 2023   # ← change year
FORECAST_VIS_MONTH = 8      # ← change month (1–12)
# ════════════════════════════════════════════════

target_month = f"{FORECAST_VIS_YEAR}-{FORECAST_VIS_MONTH:02d}"
date_strings = test_dates.strftime('%Y-%m')
matches = np.where(date_strings == target_month)[0]

if len(matches) == 0:
    print(f"❌ {target_month} not in test set.")
    print(f"   Available range: {date_strings.min()} → {date_strings.max()}")
else:
    idx   = matches[0]
    label = test_dates[idx].strftime('%B %Y')

    dmi_true = np.nanmean(y_true[idx][wi_s]) - np.nanmean(y_true[idx][ei_s])
    dmi_pred = np.nanmean(y_pred[idx][wi_s]) - np.nanmean(y_pred[idx][ei_s])
    print(f"DMI — Actual: {dmi_true:+.3f} °C | Predicted: {dmi_pred:+.3f} °C")
    phase = "Positive IOD" if dmi_true > 0.4 else ("Negative IOD" if dmi_true < -0.4 else "Neutral")
    print(f"IOD Phase (actual): {phase}")

    vmin     = np.nanmin([y_true[idx], y_pred[idx]])
    vmax_v   = np.nanmax([y_true[idx], y_pred[idx]])
    residual = y_pred[idx] - y_true[idx]
    abs_max  = np.nanmax(np.abs(residual))

    fig, axes = plt.subplots(1, 3, figsize=(MAP_FIGW * 3, MAP_FIGH))
    im0 = smooth_plot(axes[0], lon_small, lat_small, y_true[idx], cmap='RdYlBu_r', vmin=vmin,     vmax=vmax_v, ocean_mask=ocean_mask)
    im1 = smooth_plot(axes[1], lon_small, lat_small, y_pred[idx], cmap='RdYlBu_r', vmin=vmin,     vmax=vmax_v, ocean_mask=ocean_mask)
    im2 = smooth_plot(axes[2], lon_small, lat_small, residual,    cmap='bwr',       vmin=-abs_max, vmax=abs_max, ocean_mask=ocean_mask)
    axes[0].set_title(f"Actual SST ({label})")
    axes[1].set_title("ConvLSTM Forecast")
    axes[2].set_title("Residual (Predicted − Actual)")
    for ax, im in zip(axes, [im0, im1, im2]):
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_xlabel("Longitude (°E)")
        ax.set_ylabel("Latitude")
    plt.suptitle(f"1-Month Ahead SST Forecast vs Ground Truth — {label}", fontsize=15)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

---
## Extended Analysis

In [ ]:
# ── CHANGE 6 ── Cell 14: Skill Score with detailed inference output
# Persistence baseline: predict next month = current month (last frame of input)
# Skill = 1 - (MSE_model / MSE_persistence)
#   > 0  → model beats persistence
#   = 0  → model equals persistence
#   < 0  → model is worse than persistence

y_persist = X_test[:, -1, :, :, 0] * rng[np.newaxis] + sc_min[np.newaxis]
y_persist[:, ~ocean_mask] = np.nan

mse_model   = np.nanmean((y_pred - y_true) ** 2, axis=0)
mse_persist = np.nanmean((y_persist - y_true) ** 2, axis=0)
skill_score  = 1.0 - mse_model / np.where(mse_persist > 0, mse_persist, np.nan)

mean_skill  = np.nanmean(skill_score)
west_skill  = np.nanmean(skill_score[wi_s])
east_skill  = np.nanmean(skill_score[ei_s])
pct_positive = np.nanmean(skill_score > 0) * 100

fig, ax = plt.subplots(figsize=(MAP_FIGW, MAP_FIGH))
im = smooth_plot(ax, lon_small, lat_small, skill_score, cmap='RdYlGn', vmin=-1, vmax=1, ocean_mask=ocean_mask)
plt.colorbar(im, ax=ax).set_label('Skill Score')
for pole in [WEST_POLE, EAST_POLE]:
    ax.add_patch(Rectangle((pole["lon"][0], pole["lat"][0]),
                            pole["lon"][1] - pole["lon"][0], pole["lat"][1] - pole["lat"][0],
                            linewidth=1.5, edgecolor='black', facecolor='none', linestyle='--'))
ax.set(title=f'Skill Score vs Persistence  |  Mean={mean_skill:.3f}  |  {pct_positive:.1f}% of pixels beat persistence',
       xlabel='Longitude (°E)', ylabel='Latitude')
plt.tight_layout()
plt.show()

print(f"➜ Mean Skill Score (ocean): {mean_skill:.3f}")
print(f"➜ West Pole Skill         : {west_skill:.3f}")
print(f"➜ East Pole Skill         : {east_skill:.3f}")
print(f"➜ Pixels beating persistence: {pct_positive:.1f}%")
print()
print("Interpretation:")
print(f"  {'✅' if mean_skill > 0 else '❌'} {'Model adds value over persistence' if mean_skill > 0 else 'Model worse than persistence overall'}")
print(f"  {'✅' if west_skill > east_skill else '⚠️ '} {'West Pole (stable) has higher skill than East (expected)' if west_skill > east_skill else 'Unexpected: East Pole skill > West Pole'}")

In [ ]:
season_map   = {12: 'DJF', 1: 'DJF', 2: 'DJF',
                3: 'MAM', 4: 'MAM', 5: 'MAM',
                6: 'JJA', 7: 'JJA', 8: 'JJA',
                9: 'SON', 10: 'SON', 11: 'SON'}
test_months  = pd.DatetimeIndex(test_dates).month
seasons      = np.array([season_map[m] for m in test_months])
season_order = ['DJF', 'MAM', 'JJA', 'SON']

fig, axes = plt.subplots(1, 4, figsize=(MAP_FIGW * 4, MAP_FIGH))
season_rmse_vals = {}
for ax, season in zip(axes, season_order):
    mask = seasons == season
    if mask.sum() == 0:
        continue
    s_rmse = np.sqrt(np.nanmean((y_pred[mask] - y_true[mask]) ** 2, axis=0))
    vmax_s = np.nanpercentile(s_rmse, 95)
    season_rmse_vals[season] = np.nanmean(s_rmse)
    im = smooth_plot(ax, lon_small, lat_small, s_rmse, cmap='YlOrRd', vmin=0, vmax=vmax_s, ocean_mask=ocean_mask)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label('RMSE (°C)')
    ax.set_title(f"{season}  (n={mask.sum()})  mean={np.nanmean(s_rmse):.3f}°C")
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude')
plt.suptitle('Seasonal RMSE — Does the model struggle in IOD peak season (SON)?', fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

worst = max(season_rmse_vals, key=season_rmse_vals.get)
best  = min(season_rmse_vals, key=season_rmse_vals.get)
for s, v in season_rmse_vals.items():
    flag = " ← peak IOD season" if s == 'SON' else ""
    print(f"  {s}: {v:.3f} °C{flag}")
print(f"\nWorst season: {worst} ({season_rmse_vals[worst]:.3f}°C) | Best: {best} ({season_rmse_vals[best]:.3f}°C)")

In [ ]:
temporal_rmse = np.sqrt(np.nanmean((y_pred - y_true) ** 2, axis=(1, 2)))
test_dt = pd.DatetimeIndex(test_dates)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(test_dt, temporal_rmse, color='#e6550d', linewidth=1.2)
ax.fill_between(test_dt, temporal_rmse, alpha=0.25, color='#e6550d')
mean_t = np.nanmean(temporal_rmse)
ax.axhline(mean_t, color='black', linestyle='--', linewidth=1, label=f'Mean = {mean_t:.3f} °C')
# Mark top-5 worst months
top5_idx = np.argsort(temporal_rmse)[-5:][::-1]
for i in top5_idx:
    ax.annotate(test_dates[i].strftime('%Y-%m'),
                xy=(test_dt[i], temporal_rmse[i]),
                xytext=(0, 8), textcoords='offset points',
                fontsize=7, ha='center', color='#d73027')
ax.set(title='Temporal RMSE: Monthly forecast error over test period (worst months labelled)',
       xlabel='Date', ylabel='Spatial Mean RMSE (°C)')
ax.legend()
ax.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
west_true = np.nanmean(y_true[:, wi_s[0], wi_s[1]], axis=(1, 2))
west_pred = np.nanmean(y_pred[:, wi_s[0], wi_s[1]], axis=(1, 2))
east_true = np.nanmean(y_true[:, ei_s[0], ei_s[1]], axis=(1, 2))
east_pred = np.nanmean(y_pred[:, ei_s[0], ei_s[1]], axis=(1, 2))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, true_v, pred_v, title, color in [
    (axes[0], west_true, west_pred, 'West Pole', '#d73027'),
    (axes[1], east_true, east_pred, 'East Pole', '#4575b4')
]:
    ax.scatter(true_v, pred_v, alpha=0.5, s=15, color=color)
    lo = min(true_v.min(), pred_v.min())
    hi = max(true_v.max(), pred_v.max())
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, label='1:1 line')
    r2 = stats.pearsonr(true_v, pred_v)[0] ** 2
    ax.set_title(f"{title}  (R² = {r2:.3f})")
    ax.set_xlabel('Actual SST (°C)')
    ax.set_ylabel('Predicted SST (°C)')
    ax.legend()
plt.suptitle('Predicted vs Actual SST — IOD Poles', fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

In [ ]:
ocean_residuals = (y_pred - y_true)[:, ocean_mask].ravel()
ocean_residuals = ocean_residuals[~np.isnan(ocean_residuals)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(ocean_residuals, bins=80, color='#6baed6', edgecolor='white', linewidth=0.3)
ax.axvline(0,                           color='black',  linestyle='--', linewidth=1.5, label='Zero bias')
ax.axvline(ocean_residuals.mean(),      color='red',    linestyle='-',  linewidth=1.5, label=f'Mean = {ocean_residuals.mean():.3f} °C')
ax.axvline(np.median(ocean_residuals),  color='orange', linestyle='-',  linewidth=1.5, label=f'Median = {np.median(ocean_residuals):.3f} °C')
ax.set(title='Residual Distribution (Predicted − Actual)', xlabel='Error (°C)', ylabel='Count')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Skewness: {stats.skew(ocean_residuals):.3f} | Kurtosis: {stats.kurtosis(ocean_residuals):.3f}")

In [ ]:
equatorial = (lat_small >= -5) & (lat_small <= 5)
true_eq    = y_true[:, equatorial, :]
true_eq_m  = np.nanmean(true_eq, axis=1)

test_month_idx = pd.DatetimeIndex(test_dates).month - 1
clim_eq = np.array([true_eq_m[test_month_idx == m].mean(axis=0)
                    if (test_month_idx == m).any() else np.zeros(true_eq_m.shape[1])
                    for m in range(12)])
anom_eq = true_eq_m - clim_eq[test_month_idx]

vlim = np.nanpercentile(np.abs(anom_eq), 97)
fig, ax = plt.subplots(figsize=(14, 5))
im = ax.pcolormesh(lon_small, np.arange(len(test_dates)), anom_eq,
                   cmap='RdBu_r', vmin=-vlim, vmax=vlim, shading='auto')
plt.colorbar(im, ax=ax).set_label('SST Anomaly (°C)')
tick_step = max(1, len(test_dates) // 10)
ax.set_yticks(np.arange(0, len(test_dates), tick_step))
ax.set_yticklabels([test_dates[i].strftime('%Y-%m') for i in range(0, len(test_dates), tick_step)], fontsize=8)
ax.set(title='Hovmöller Diagram: Equatorial SST Anomaly (5°S–5°N)',
       xlabel='Longitude (°E)', ylabel='Date')
plt.tight_layout()
plt.show()

In [ ]:
# ── CHANGE 7 ── Cell 20: DMI time series with direct year entry
# ════════════════════════════════════════════════
#  FILTER DMI DISPLAY TO A SPECIFIC YEAR RANGE
DMI_DISPLAY_START = None   # ← set to a year e.g. 2015, or None for full test period
DMI_DISPLAY_END   = None   # ← set to a year e.g. 2023, or None for full test period
# ════════════════════════════════════════════════

clim_west = np.array([west_true[pd.DatetimeIndex(test_dates).month == m].mean()
                      if (pd.DatetimeIndex(test_dates).month == m).any() else 0.0
                      for m in range(1, 13)])
clim_east = np.array([east_true[pd.DatetimeIndex(test_dates).month == m].mean()
                      if (pd.DatetimeIndex(test_dates).month == m).any() else 0.0
                      for m in range(1, 13)])

m_idx = pd.DatetimeIndex(test_dates).month - 1
dmi_true_ts = (west_true - clim_west[m_idx]) - (east_true - clim_east[m_idx])
dmi_pred_ts = (west_pred - clim_west[m_idx]) - (east_pred - clim_east[m_idx])

# Apply optional year filter
plot_mask = np.ones(len(test_dt), dtype=bool)
if DMI_DISPLAY_START is not None:
    plot_mask &= (test_dt.year >= DMI_DISPLAY_START)
if DMI_DISPLAY_END is not None:
    plot_mask &= (test_dt.year <= DMI_DISPLAY_END)

dt_plot   = test_dt[plot_mask]
true_plot = dmi_true_ts[plot_mask]
pred_plot = dmi_pred_ts[plot_mask]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(dt_plot, true_plot, color='black',   linewidth=1.5, label='Actual DMI')
ax.plot(dt_plot, pred_plot, color='#e6550d', linewidth=1.5, linestyle='--', label='Predicted DMI')
ax.fill_between(dt_plot, true_plot, 0, where=(true_plot >= 0), color='#d73027', alpha=0.25, label='Positive IOD')
ax.fill_between(dt_plot, true_plot, 0, where=(true_plot  < 0), color='#4575b4', alpha=0.25, label='Negative IOD')
ax.axhline(0,    color='gray',    linestyle='--', linewidth=0.8)
ax.axhline( 0.5, color='#d73027', linestyle=':',  linewidth=0.8, alpha=0.6)
ax.axhline(-0.5, color='#4575b4', linestyle=':',  linewidth=0.8, alpha=0.6)
ax.set(title='Actual vs Predicted DMI Anomaly Time Series',
       xlabel='Date', ylabel='DMI (°C)')
ax.legend()
ax.grid(color='gray', linestyle='--', linewidth=0.4, alpha=0.4)
plt.tight_layout()
plt.show()

r_dmi = stats.pearsonr(dmi_true_ts, dmi_pred_ts)[0]
print(f"DMI Pearson r = {r_dmi:.3f}")
print(f"Positive IOD months (actual >0.5): {(dmi_true_ts > 0.5).sum()} | predicted: {(dmi_pred_ts > 0.5).sum()}")
print(f"Negative IOD months (actual <−0.5): {(dmi_true_ts < -0.5).sum()} | predicted: {(dmi_pred_ts < -0.5).sum()}")

In [ ]:
nan_frac = np.isnan(grid_small).mean(axis=0)

fig, ax = plt.subplots(figsize=(MAP_FIGW, MAP_FIGH))
im = smooth_plot(ax, lon_small, lat_small, nan_frac * 100, cmap='Reds', vmin=0, vmax=100, ocean_mask=ocean_mask)
plt.colorbar(im, ax=ax).set_label('% Missing (NaN)')
ax.set(title='Missing Data Heatmap (Pre-Imputation) — Land Mask Verification',
       xlabel='Longitude (°E)', ylabel='Latitude')
plt.tight_layout()
plt.show()
print(f"Ocean pixels with 0% missing : {(nan_frac == 0).sum()}")
print(f"Pixels with >90% missing (land): {(nan_frac > 0.9).sum()}")

In [ ]:
# ── CHANGE 8 ── Cell 22: ACF with lag explanation annotations
# Lag = time shift (months). Lag-12 bar height justifies seq_len=12.
# High lag-12 ACF means SST this month correlates with SST 1 year ago.

west_full = np.nanmean(grid_modern[:, pole_mask(lat_small, lon_small, WEST_POLE)[0],
                                       pole_mask(lat_small, lon_small, WEST_POLE)[1]], axis=(1, 2))
east_full = np.nanmean(grid_modern[:, pole_mask(lat_small, lon_small, EAST_POLE)[0],
                                       pole_mask(lat_small, lon_small, EAST_POLE)[1]], axis=(1, 2))

max_lag = 36
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, series, title, color in [
    (axes[0], west_full, 'West Pole ACF', '#d73027'),
    (axes[1], east_full, 'East Pole ACF', '#4575b4')
]:
    acf_vals = [pd.Series(series).autocorr(lag=l) for l in range(max_lag + 1)]
    ax.bar(range(max_lag + 1), acf_vals, color=color, alpha=0.7)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(12, color='black', linestyle='--', linewidth=1.2, label=f'lag=12 (annual)  ACF={acf_vals[12]:.2f}')
    ax.axvline(24, color='gray',  linestyle=':',  linewidth=0.8, label=f'lag=24 (biennial) ACF={acf_vals[24]:.2f}')
    ax.axhline(1.96/np.sqrt(len(series)), color='green', linestyle=':', linewidth=0.8, label='95% CI')
    ax.axhline(-1.96/np.sqrt(len(series)), color='green', linestyle=':', linewidth=0.8)
    ax.set(title=title, xlabel='Lag (months)  [1 bar = 1 month shift]',
           ylabel='Autocorrelation', ylim=(-0.3, 1.05))
    ax.legend(fontsize=8)
    print(f"{title}: lag-1={acf_vals[1]:.3f}  lag-12={acf_vals[12]:.3f}  lag-24={acf_vals[24]:.3f}")
plt.suptitle('ACF — Justification for seq_len=12  |  Bars beyond green line are statistically significant', fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

In [ ]:
# ── CHANGE 9 + 10 ── Cell 23: Unified DMI Forecaster
# Works for: historical record, test set, AND future dates
# Just change TARGET_YEAR and TARGET_MONTH below.
#
# ════════════════════════════════════════════════
TARGET_YEAR  = 2028   # ← enter any year
TARGET_MONTH = 1      # ← enter any month (1–12)
# ════════════════════════════════════════════════

target_dt     = pd.Timestamp(year=TARGET_YEAR, month=TARGET_MONTH, day=1)
target_str    = target_dt.strftime('%Y-%m')
label         = target_dt.strftime('%B %Y')
last_known_dt = pd.Timestamp(dates_modern[-1])
months_ahead  = (target_dt.year - last_known_dt.year) * 12 + \
                (target_dt.month - last_known_dt.month)
date_strings  = test_dates.strftime('%Y-%m')
in_test       = np.where(date_strings == target_str)[0]

future_pred_celsius = None
dmi_val = None

# ── Case 1: within test set ──────────────────────
if len(in_test) > 0:
    idx      = in_test[0]
    dmi_true = np.nanmean(y_true[idx][wi_s]) - np.nanmean(y_true[idx][ei_s])
    dmi_pred = np.nanmean(y_pred[idx][wi_s]) - np.nanmean(y_pred[idx][ei_s])
    dmi_val  = dmi_pred
    future_pred_celsius = y_pred[idx]
    source_label = "Test Set (model already predicted this)"
    print(f"📅 {label} — found in test set")
    print(f"   Actual DMI    : {dmi_true:+.3f} °C")
    print(f"   Predicted DMI : {dmi_pred:+.3f} °C")

# ── Case 2: future (autoregressive) ─────────────
elif months_ahead > 0:
    source_label = f"Autoregressive Forecast ({months_ahead} months ahead)"
    print(f"🔮 {label} — {months_ahead} months ahead, running autoregressive rollout...")

    window = scaled[-seq_len:][..., np.newaxis][np.newaxis]  # (1,12,H,W,1)
    for step in range(months_ahead):
        pred_s = np.clip(model.predict(window, verbose=0), 0.0, 1.0)
        window = np.concatenate([window[:, 1:, ...], pred_s[:, np.newaxis, ...]], axis=1)
        if (step + 1) % 12 == 0:
            print(f"   ...{step+1} months done", end="\r")

    future_pred_celsius = pred_s[0, :, :, 0] * rng + sc_min
    future_pred_celsius[~ocean_mask] = np.nan
    dmi_val = np.nanmean(future_pred_celsius[wi_s]) - np.nanmean(future_pred_celsius[ei_s])
    print(f"\n✅ Forecast complete")
    print(f"   Forecasted DMI: {dmi_val:+.3f} °C")

# ── Case 3: historical (training period) ────────
else:
    hist_mask = dates_modern == target_dt
    if hist_mask.any():
        t_idx    = np.where(hist_mask)[0][0]
        g_raw    = grid_small[t_idx]
        dmi_val  = np.nanmean(g_raw[wi_s]) - np.nanmean(g_raw[ei_s])
        future_pred_celsius = g_raw
        source_label = "Historical Record (training period)"
        print(f"📚 {label} — historical record")
        print(f"   Observed DMI: {dmi_val:+.3f} °C")
    else:
        print(f"❌ {label} not found in any available data.")
        dmi_val = None

# ── IOD Phase Interpretation ─────────────────────
if dmi_val is not None:
    print()
    if dmi_val > 0.5:
        print("   🔴 POSITIVE IOD — warm west, cool east")
        print("      → Enhanced Indian monsoon | Australian drought risk | E.Africa floods")
    elif dmi_val < -0.5:
        print("   🔵 NEGATIVE IOD — cool west, warm east")
        print("      → Reduced Indian monsoon | Floods in Australia/Indonesia")
    else:
        print("   ⚪ NEUTRAL IOD — no significant dipole signal")

    if months_ahead > 12:
        print(f"\n   ⚠️  Reliability note: {months_ahead} months ahead — seasonal structure only, extreme events unreliable")

# ── Spatial Plot ─────────────────────────────────
if future_pred_celsius is not None:
    fig, ax = plt.subplots(figsize=(MAP_FIGW, MAP_FIGH))
    im = smooth_plot(ax, lon_small, lat_small, future_pred_celsius, cmap='RdYlBu_r', ocean_mask=ocean_mask)
    plt.colorbar(im, ax=ax).set_label('SST (°C)')
    for pole in [WEST_POLE, EAST_POLE]:
        ax.add_patch(Rectangle((pole["lon"][0], pole["lat"][0]),
                                pole["lon"][1] - pole["lon"][0],
                                pole["lat"][1] - pole["lat"][0],
                                linewidth=1.5, edgecolor='black',
                                facecolor='none', linestyle='--'))
    dmi_str = f"DMI = {dmi_val:+.3f}°C" if dmi_val is not None else ""
    ax.set(title=f"{label}  |  {source_label}  |  {dmi_str}",
           xlabel='Longitude (°E)', ylabel='Latitude')
    plt.tight_layout()
    plt.show()